# In-Context Learning + Attention Head Analysis
## GPT-2 XL — Repetition Pattern

Can GPT-2 XL learn a simple repetition rule from examples in the prompt? And can we find the attention heads that implement this by looking back at the source word?

Specifically we will make a new simple task: We will (1) cue the network with a specific one-token word (e.g., dog), (2) give it an arbitrary symbol — **which you will choose** — that we designate means "copy the preceding word three times."

For example, if you choose `>>>` as your symbol:

```
dog >>> dog dog dog
```

The model has never seen this convention before — it has to learn what your symbol means purely from examples in the prompt. This is obviously a silly task, but it allows us to understand basic principles of how Transformers integrate new information from the prompt so successfully. We'll then look *inside* the model to find the attention heads that implement this by looking back at the source word.


In [1]:
#LOAD GPT2-XL. Takes a minute to load.

import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

plt.rcParams['figure.dpi'] = 150

torch_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {torch_device}")

model_name = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    pad_token_id=tokenizer.eos_token_id,
    output_attentions=True
).to(torch_device)
model.eval()
print(f"Loaded {model_name} ({sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters)")


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded gpt2-xl (1558M parameters)


In [2]:
# ============================================================
# CHOOSE YOUR TASK SIGNAL
# ============================================================
# Pick a single token that will mean "copy this word three times."
# It should be something arbitrary — no natural connection to copying.
#
# Some suggestions (but feel free to try your own!):
#   >>>    (Python prompt — no link to "repeat")
#   ::     (C++ scope operator)
#   |||    (looks like a barrier)
#   ##     (comment marker)
#   @@     (email symbol)
#   ***    (markdown bold)
#   +++    (diff marker)

SEPARATOR = "@@@"  # <-- CHANGE THIS TO WHATEVER YOU WANT

# Verify it's a single token (critical for the attention analysis to work smoothly as coded)
test_str = f" {SEPARATOR}"
sep_ids = tokenizer.encode(test_str)
sep_tokens = tokenizer.convert_ids_to_tokens(sep_ids)

print(f"Your separator: '{SEPARATOR}'")
print(f"Tokenization of ' {SEPARATOR}': {sep_tokens}  ({len(sep_tokens)} token(s))")
print()

if len(sep_tokens) <= 2:  # space token + separator token
    print(f"✅ '{SEPARATOR}' is a single token — good to go!")
    print(f"\nYour task: the model will learn that")
    print(f"   word {SEPARATOR} word word word")
    print(f"means 'copy the word three times.'")
else:
    print(f"⚠️  '{SEPARATOR}' splits into {len(sep_tokens)} tokens.")
    print(f"    This will make the attention analysis harder to interpret.")
    print(f"    Try a different separator (>>>, ::, ##, etc.)")


Your separator: '@@@'
Tokenization of ' @@@': ['Ġ@', '@@']  (2 token(s))

✅ '@@@' is a single token — good to go!

Your task: the model will learn that
   word @@@ word word word
means 'copy the word three times.'


## Step 1: Does it learn the pattern?

First, let's test whether GPT-2 XL can learn a simple tripling rule from examples.


In [3]:
# ============================================================
# TEST: Can GPT-2 XL learn the repetition pattern with YOUR separator?
# ============================================================

prompts = {
    "0 examples": f"moon {SEPARATOR}",
    "1 example":  f"cat {SEPARATOR} cat cat cat\nmoon {SEPARATOR}",
    "2 examples": f"cat {SEPARATOR} cat cat cat\ndog {SEPARATOR} dog dog dog\nmoon {SEPARATOR}",
    "3 examples": f"cat {SEPARATOR} cat cat cat\ndog {SEPARATOR} dog dog dog\nsun {SEPARATOR} sun sun sun\nmoon {SEPARATOR}",
}

print(f"=== Can GPT-2 XL learn: word {SEPARATOR} word word word? ===\n")
for label, prompt in prompts.items():
    inputs = tokenizer(prompt, return_tensors='pt').to(torch_device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    continuation = generated[len(prompt):].strip().split('\n')[0]
    correct = "moon moon moon" in generated
    marker = "✅" if correct else "❌"
    print(f"  {label:15s} | moon {SEPARATOR} {continuation:30s} {marker}")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== Can GPT-2 XL learn: word @@@ word word word? ===



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  0 examples      | moon @@@ @                              ❌


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  1 example       | moon @@@ moon moon moon                 ✅


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  2 examples      | moon @@@ moon moon moon                 ✅
  3 examples      | moon @@@ moon moon moon                 ✅


In [4]:
# Also test with a word NOT in the examples (true generalization)
prompt_novel = f"cat {SEPARATOR} cat cat cat\ndog {SEPARATOR} dog dog dog\nsun {SEPARATOR} sun sun sun\nriver {SEPARATOR}"

inputs = tokenizer(prompt_novel, return_tensors='pt').to(torch_device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=10, do_sample=False)
generated = tokenizer.decode(output[0], skip_special_tokens=True)
continuation = generated[len(prompt_novel):].strip().split('\n')[0]
print(f"Novel word test: river {SEPARATOR} {continuation}")
print(f"Expected: river river river")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Novel word test: river @@@ river river river
Expected: river river river


## Step 2: Look inside — which attention heads implement this?

When the model generates "moon" at the output, it *must* attend back to "moon" in the input — there's nowhere else to get that token. Let's find the heads that do this.


### Averaging Over Multiple Test Words

A head that attends to "moon" might just like that particular token. To find heads that *consistently* attend to whichever word is in the copy position, we run the same analysis across many different test words and **average** the attention scores. Heads that reliably look back at the to-be-copied word — regardless of what that word is — are the true induction heads.


In [5]:
# ============================================================
# RUN ATTENTION ANALYSIS OVER MANY TEST WORDS
# ============================================================
# For each test word, we build the same prompt structure:
#   cat >>> cat cat cat
#   dog >>> dog dog dog
#   sun >>> sun sun sun
#   [TEST] >>>
# and measure how much each head attends from the LAST token position
# to the TEST word vs. the other input words (cat, dog, sun).
#
# Note: ">>>" may be split into multiple tokens by GPT-2's BPE tokenizer.
# We find the LAST token in the sequence as the generation position.

def find_token(tokens, target, start_from=0):
    """Find index of token containing target (handles GPT-2 prefix chars)."""
    for i in range(start_from, len(tokens)):
        clean = tokens[i].replace('\u0120', '').replace('\u010a', '')
        if clean == target:
            return i
    return None

def find_all_tokens(tokens, target):
    """Find ALL positions matching target."""
    results = []
    for i, t in enumerate(tokens):
        clean = t.replace('\u0120', '').replace('\u010a', '')
        if clean == target:
            results.append(i)
    return results

def find_sep_positions(tokens):
    """Find the START position of each '>>>' in the token sequence.
    >>> may be split into multiple tokens (e.g., 'X', 'OX', 'O').
    We find each occurrence by looking for the decoded substring."""
    positions = []
    text_so_far = ''
    for i, t in enumerate(tokens):
        text_so_far += t.replace('\u0120', ' ').replace('\u010a', '\n')
        if text_so_far.rstrip().endswith('>>>'):
            positions.append(i)  # last token of this >>>
    return positions

# Test words — varied enough that no single token preference can fake the pattern
test_words = [
    'moon', 'river', 'fish', 'tree', 'lamp', 'snow', 'fire', 'rain',
    'king', 'door', 'hand', 'ship', 'bird', 'gold', 'stone', 'milk',
    'hill', 'star', 'bone', 'seed', 'rope', 'bell', 'dust', 'cake',
    'wine', 'iron', 'coal', 'silk', 'wolf', 'drum', 'salt', 'clay',
]
context_words = ['cat', 'dog', 'sun']  # always the same 3 examples

# First pass: figure out how >>> is tokenized and get model dimensions
sample_prompt = f"cat {SEPARATOR} cat cat cat\ndog {SEPARATOR} dog dog dog\nsun {SEPARATOR} sun sun sun\nmoon {SEPARATOR}"
sample_inputs = tokenizer(sample_prompt, return_tensors='pt').to(torch_device)
sample_tokens = tokenizer.convert_ids_to_tokens(sample_inputs['input_ids'][0])
print(f"Tokenization of prompt:")
for j, t in enumerate(sample_tokens):
    print(f"  [{j:2d}] {t}")

with torch.no_grad():
    sample_out = model(**sample_inputs)
n_layers = len(sample_out.attentions)
n_heads = sample_out.attentions[0].shape[1]
print(f"\nModel has {n_layers} layers x {n_heads} heads = {n_layers * n_heads} total heads")

# Find the >>> positions
sep_positions = find_sep_positions(sample_tokens)
print(f">>> end-positions: {sep_positions}")
print(f"Last position (generation point): {len(sample_tokens) - 1}")

# Accumulate: for each head, how much does it attend to the TEST word vs. context words?
attn_to_test_word = np.zeros((n_layers, n_heads, len(test_words)))
attn_to_context_avg = np.zeros((n_layers, n_heads, len(test_words)))

for w_idx, test_word in enumerate(test_words):
    prompt = f"cat {SEPARATOR} cat cat cat\ndog {SEPARATOR} dog dog dog\nsun {SEPARATOR} sun sun sun\n{test_word} {SEPARATOR}"
    inputs = tokenizer(prompt, return_tensors='pt').to(torch_device)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    with torch.no_grad():
        outputs = model(**inputs)

    attention = outputs.attentions

    # The generation position is the LAST token in the sequence
    last_pos = len(tokens) - 1

    # Find the test word (should be right before the last >>>)
    test_idx = find_token(tokens, test_word)

    # Find first occurrence of each context word
    ctx_indices = []
    for cw in context_words:
        idx = find_token(tokens, cw)
        if idx is not None:
            ctx_indices.append(idx)

    if test_idx is None:
        print(f"  WARNING: could not find '{test_word}' in tokens: {tokens}")
        continue

    for layer in range(n_layers):
        for head in range(n_heads):
            attn = attention[layer][0, head, last_pos, :].cpu().numpy()
            attn_to_test_word[layer, head, w_idx] = attn[test_idx]
            if ctx_indices:
                attn_to_context_avg[layer, head, w_idx] = np.mean([attn[ci] for ci in ctx_indices])

    if w_idx % 8 == 0:
        print(f"  Processed {w_idx+1}/{len(test_words)}: {test_word} (position {test_idx}, generating from {last_pos})")

# Average across all test words
mean_attn_to_test = attn_to_test_word.mean(axis=2)
mean_attn_to_context = attn_to_context_avg.mean(axis=2)

# Preference: how much MORE does the head attend to the test word than context words?
preference = mean_attn_to_test - mean_attn_to_context

print(f"\nDone! Averaged over {len(test_words)} test words.")


Tokenization of prompt:
  [ 0] cat
  [ 1] Ġ@
  [ 2] @@
  [ 3] Ġcat
  [ 4] Ġcat
  [ 5] Ġcat
  [ 6] Ċ
  [ 7] dog
  [ 8] Ġ@
  [ 9] @@
  [10] Ġdog
  [11] Ġdog
  [12] Ġdog
  [13] Ċ
  [14] sun
  [15] Ġ@
  [16] @@
  [17] Ġsun
  [18] Ġsun
  [19] Ġsun
  [20] Ċ
  [21] moon
  [22] Ġ@
  [23] @@

Model has 48 layers x 25 heads = 1200 total heads
>>> end-positions: []
Last position (generation point): 23
  Processed 1/32: moon (position 21, generating from 23)
  Processed 9/32: king (position 21, generating from 23)
  Processed 17/32: hill (position 21, generating from 23)
  Processed 25/32: wine (position 21, generating from 23)

Done! Averaged over 32 test words.


In [6]:
# ============================================================
# RANK HEADS BY CONSISTENT PREFERENCE FOR THE TEST WORD
# ============================================================

print("Top 15 heads that consistently attend to the WORD TO BE COPIED:\n")
print(f"{'Layer':>5} {'Head':>4} | {'→test word':>11} {'→context avg':>13} | {'preference':>10} {'std across words':>16}")
print("-" * 80)

flat_idx = np.argsort(preference.ravel())[::-1]
top_heads = []

for rank in range(15):
    idx = flat_idx[rank]
    l, h = np.unravel_index(idx, (n_layers, n_heads))
    test_mean = mean_attn_to_test[l, h]
    ctx_mean = mean_attn_to_context[l, h]
    pref = preference[l, h]
    # Standard deviation across test words (low = consistent)
    std = attn_to_test_word[l, h, :].std()
    top_heads.append((l, h))
    print(f"L{l:>3} H{h:>3} | {test_mean:>11.4f} {ctx_mean:>13.4f} | {pref:>10.4f} {std:>16.4f}")

best_l, best_h = top_heads[0]
print(f"\nBest induction head candidate: Layer {best_l}, Head {best_h}")


Top 15 heads that consistently attend to the WORD TO BE COPIED:

Layer Head |  →test word  →context avg | preference std across words
--------------------------------------------------------------------------------
L 26 H 20 |      0.7402        0.0026 |     0.7376           0.4275
L 29 H 19 |      0.7402        0.0027 |     0.7375           0.4286
L 28 H 11 |      0.7320        0.0034 |     0.7286           0.4231
L 24 H  5 |      0.6840        0.0034 |     0.6806           0.4031
L 38 H 24 |      0.6703        0.0144 |     0.6559           0.4119
L 22 H 21 |      0.6677        0.0192 |     0.6485           0.3898
L 24 H 16 |      0.6694        0.0260 |     0.6434           0.4205
L 26 H 22 |      0.6448        0.0153 |     0.6295           0.3942
L 25 H 18 |      0.6535        0.0302 |     0.6233           0.4124
L 26 H  4 |      0.6131        0.0349 |     0.5781           0.3735
L 40 H 23 |      0.5930        0.0355 |     0.5575           0.3690
L 29 H 23 |      0.5867        0.0452

In [7]:
# ============================================================
#  Does this head generalize? Test with a different word.
# ============================================================

prompt2 = f"cat {SEPARATOR} cat cat cat\ndog {SEPARATOR} dog dog dog\nsun {SEPARATOR} sun sun sun\nriver {SEPARATOR}"
inputs2 = tokenizer(prompt2, return_tensors='pt').to(torch_device)
with torch.no_grad():
    outputs2 = model(**inputs2)

tokens2 = tokenizer.convert_ids_to_tokens(inputs2['input_ids'][0])
attn2 = outputs2.attentions

river_idx = find_token(tokens2, 'river')
last_pos2 = len(tokens2) - 1

cat_idx2 = find_token(tokens2, 'cat')
dog_idx2 = find_token(tokens2, 'dog')
sun_idx2 = find_token(tokens2, 'sun')

print(f"Same head (L{best_l} H{best_h}) on a NEW test word 'river':\n")
print(f"  → river:  {attn2[best_l][0, best_h, last_pos2, river_idx].cpu().item():.4f}")
print(f"  → cat:    {attn2[best_l][0, best_h, last_pos2, cat_idx2].cpu().item():.4f}")
print(f"  → dog:    {attn2[best_l][0, best_h, last_pos2, dog_idx2].cpu().item():.4f}")
print(f"  → sun:    {attn2[best_l][0, best_h, last_pos2, sun_idx2].cpu().item():.4f}")

print(f"\nIf this head is a true 'induction head', it should attend to 'river'")
print(f"just as strongly as it attended to 'moon' — because the ROLE is the")
print(f"same (word-to-be-copied), even though the CONTENT is different.")
print(f"This is the same role/filler separation we saw in the ESBN!")


Same head (L26 H20) on a NEW test word 'river':

  → river:  0.9863
  → cat:    0.0000
  → dog:    0.0002
  → sun:    0.0092

If this head is a true 'induction head', it should attend to 'river'
just as strongly as it attended to 'moon' — because the ROLE is the
same (word-to-be-copied), even though the CONTENT is different.
This is the same role/filler separation we saw in the ESBN!


## Visualize the Attention Patterns Using BertViz


How to:
1. Select a particular layer using the drop down menu.
2.  Hover your cursor over a token to see the attention pattern.
3.  Look at what happens when you hover over the SEPARATOR token - does the model look back at the source word?
5. Compare an early layer (e.g., 1-5) to a later layer (e.g., 15+).

In [ ]:
!pip install bertviz
from transformers import AutoTokenizer, AutoModel, utils
from bertviz import model_view, head_view, neuron_view


In [12]:
from transformers import AutoModel, AutoTokenizer

# Configure model to return attention values
model_viz = AutoModel.from_pretrained(model_name, output_attentions=True)
tokenizer_viz = AutoTokenizer.from_pretrained(model_name)

# Use the 3-example prompt with your chosen separator
prompt_viz = f"cat {SEPARATOR} cat cat cat\ndog {SEPARATOR} dog dog dog\nsun {SEPARATOR} sun sun sun\nmoon {SEPARATOR}"
inputs = tokenizer_viz.encode(prompt_viz, return_tensors='pt')

outputs = model_viz(inputs)
attention = outputs[-1]
tokens = tokenizer_viz.convert_ids_to_tokens(inputs[0])

print(f"Visualizing attention for: {prompt_viz}")

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Visualizing attention for: cat @@@ cat cat cat
dog @@@ dog dog dog
sun @@@ sun sun sun
moon @@@


In [14]:
!pip install bertviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.6 MB/s eta 0:00:00


In [15]:
from bertviz import head_view

# This takes a minute to load!
# Visualize attention with BertViz head_view
head_view(attention, tokens)

# Select a particular layer using the drop down menu.
# Hover your cursor over a token to see the attention pattern.
# Look at what happens when you hover over the SEPARATOR token
# and the word right before it — does the model look back at the source word?
# Compare an early layer (e.g., 1-5) to a later layer (e.g., 15+). Try to find one layer with heads that not attend to the token that needs to be copied and one layer that does.

Output hidden; open in https://colab.research.google.com to view.

## What You're Seeing

If the analysis worked, you found that a subset of attention heads that:

1. **Attend from the final ">>>" back to "moon"** — the word that needs to be copied into the output
2. **Prefer "moon" over other input words** — they don't just attend to any noun; they specifically attend to the one that appears right before ">>>" in the current incomplete line
3. **Generalize to new words** — the same head attends to "river" when we swap the test word

These are **induction heads** (Olsson et al., 2022). They implement in-context learning by:
1. Recognizing the pattern: "word → word word word"
2. Finding the current word that needs to be completed: "moon →"
3. Looking back to copy "moon" into the output
